# RAG Pipeline — Fine-Tuned Qwen 2.5 Coder 7B

This notebook implements an **error-driven RAG pipeline** using a **fine-tuned** Qwen 2.5 Coder 7B model.

## Model
- **Fine-tuned model**: `H4miid/qwen2_5_coder_7b_merged_f16.gguf` (HuggingFace)
- **Format**: GGUF (quantized for efficient inference)
- **Backend**: `llama-cpp-python` via LangChain

## Pipeline Overview
1. **Load Dataset** — Split into train/test using `train_test_split`
2. **Load Failed Samples** — From Pre_Test smoke report
3. **RAG Retrieval** — ChromaDB + BGE embeddings + reranking
4. **LangChain Chain** — Structured RAG chain for code fixing
5. **Inference** — Generate corrected code
6. **Evaluation** — Syntax check + runtime test

## GPU / vast.ai Setup
This notebook is optimized for GPU instances:
- **GPU mode**: Automatically detects GPU and offloads all layers
- **CPU mode**: Falls back to CPU if no GPU detected
- **vast.ai**: Set `INSTALL_CUDA_LLAMA=True` in Section 0 to install CUDA-enabled llama-cpp

**Recommended vast.ai specs**: RTX 3090/4090 or A100 with 24GB+ VRAM

## Dependencies
- `llama-cpp-python` — GGUF model inference (with CUDA for GPU)
- `langchain` — RAG chain orchestration
- `chromadb` — Vector database
- `sentence-transformers` — Embeddings & reranking
- `huggingface-hub` — Model download

## 1 — Imports

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STANDARD LIBRARY
# ══════════════════════════════════════════════════════════════════════════════
import json
import os
import re
import time
import random
import subprocess
import sys
import py_compile
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter

# ══════════════════════════════════════════════════════════════════════════════
# DATA & ML
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# VECTOR DB & EMBEDDINGS
# ══════════════════════════════════════════════════════════════════════════════
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import CrossEncoder

# ══════════════════════════════════════════════════════════════════════════════
# LANGCHAIN
# ══════════════════════════════════════════════════════════════════════════════
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_community.llms import LlamaCpp
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# ══════════════════════════════════════════════════════════════════════════════
# HUGGINGFACE
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download, login
from dotenv import load_dotenv

print("✓ All imports successful.")

## 0 — GPU Setup (for vast.ai / cloud instances)

Detect available GPUs and ensure `llama-cpp-python` is compiled with CUDA support.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GPU DETECTION
# ══════════════════════════════════════════════════════════════════════════════
import subprocess
import shutil

def check_gpu():
    """Detect available GPUs and VRAM."""
    print("=" * 60)
    print("GPU DETECTION")
    print("=" * 60)
    
    # Check for nvidia-smi
    if shutil.which("nvidia-smi"):
        try:
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", 
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                gpus = result.stdout.strip().split("\n")
                print(f"\n✓ Found {len(gpus)} GPU(s):\n")
                for i, gpu in enumerate(gpus):
                    parts = [p.strip() for p in gpu.split(",")]
                    if len(parts) >= 4:
                        name, total, free, driver = parts
                        print(f"  GPU {i}: {name}")
                        print(f"         VRAM: {int(total)/1024:.1f} GB total, {int(free)/1024:.1f} GB free")
                        print(f"         Driver: {driver}")
                return True
        except Exception as e:
            print(f"Error querying GPU: {e}")
    
    # Check PyTorch CUDA
    try:
        import torch
        if torch.cuda.is_available():
            print(f"\n✓ PyTorch CUDA available")
            print(f"  Device count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
                props = torch.cuda.get_device_properties(i)
                print(f"         VRAM: {props.total_memory / 1024**3:.1f} GB")
            return True
        else:
            print("\n⚠ PyTorch CUDA not available")
    except ImportError:
        print("\n⚠ PyTorch not installed")
    
    print("\n✗ No GPU detected - will run on CPU (slower)")
    return False

GPU_AVAILABLE = check_gpu()
print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INSTALL llama-cpp-python WITH CUDA SUPPORT
# ══════════════════════════════════════════════════════════════════════════════
# Run this cell ONCE on vast.ai or other GPU instances
# Takes ~5-10 minutes to compile

# Uninstall existing CPU version
!pip uninstall llama-cpp-python -y

# Install with CUDA support (works for RTX 30xx, 40xx, A100, etc.)
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir

print("\n✓ llama-cpp-python installed with CUDA support")
print("  Restart kernel before continuing")

## 2 — Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PATHS
# ══════════════════════════════════════════════════════════════════════════════
DATA_PATH       = Path("../Datasets/final_dataset.json").resolve()
CHROMA_DIR      = str(Path("../VectorDB/chroma_library_docs").resolve())
OUT_DIR         = Path("RAG_outputs/RAG_with_SFT").resolve()
MODEL_DIR       = Path("../Models").resolve()
COLLECTION_NAME = "library_docs"

# Smoke report from SFT model Pre_Test
SMOKE_REPORT_PATH = Path(
    r"../Fine-Tuning/Qwen/Fine-Tun_Results/fine_tuned_eval_runtime_A/smoke_report.json"
).resolve()

# ══════════════════════════════════════════════════════════════════════════════
# MODEL CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
HF_MODEL_REPO   = "H4miid/qwen2_5_coder_7b_merged_f16.gguf"
HF_MODEL_FILE   = "qwen2_5_coder_7b_merged_f16.gguf"
LOCAL_MODEL_PATH = MODEL_DIR / HF_MODEL_FILE

# LLM settings
N_CTX           = 8192      # context window
N_GPU_LAYERS    = -1 if GPU_AVAILABLE else 0  # -1 = all GPU layers, 0 = CPU only
TEMPERATURE     = 0.0       # deterministic output
MAX_TOKENS      = 4096      # max output tokens

# ══════════════════════════════════════════════════════════════════════════════
# DATASET SPLIT
# ══════════════════════════════════════════════════════════════════════════════
SEED            = 42
TEST_SIZE       = 0.15      # 15% for evaluation (same as baseline)

# ══════════════════════════════════════════════════════════════════════════════
# RAG SETTINGS
# ══════════════════════════════════════════════════════════════════════════════
N_RETRIEVE          = 10    # candidates per library from bi-encoder
N_RERANK            = 3     # top-k after cross-encoder reranking
MAX_QUERY           = 500   # max chars of error text used as RAG query
MAX_CTX_CHARS       = 3000  # max total chars of raw RAG context
MIN_RERANKER_SCORE  = 0.0   # score gate for cross-encoder

# Reranker model
RERANKER_MODEL = "BAAI/bge-reranker-base"

# ══════════════════════════════════════════════════════════════════════════════
# RUNTIME EVAL SETTINGS
# ══════════════════════════════════════════════════════════════════════════════
TIMEOUT         = 300       # seconds per script
FORCE_CPU       = False     # set True to disable GPU for evaluation

# Create output directory
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset         : {DATA_PATH}")
print(f"Smoke report    : {SMOKE_REPORT_PATH}")
print(f"ChromaDB        : {CHROMA_DIR}")
print(f"Output dir      : {OUT_DIR}")
print(f"Model repo      : {HF_MODEL_REPO}")
print(f"Local model     : {LOCAL_MODEL_PATH}")
print(f"Test size       : {TEST_SIZE} (seed={SEED})")
print(f"GPU layers      : {N_GPU_LAYERS} ({'GPU' if GPU_AVAILABLE else 'CPU'} mode)")

## 3 — Load Dataset & Split

In [ ]:
# Load the full dataset
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset_full = json.load(f)

print(f"Total samples: {len(dataset_full)}")

# Train/test split
train_set, test_set = train_test_split(
    dataset_full,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True
)

print(f"Train set: {len(train_set)} samples")
print(f"Test set : {len(test_set)} samples")

# Build lookup for original sample IDs (for comparison later)
sample_lookup = {sample["id"]: sample for sample in dataset_full}

## 4 — Load Smoke Report & Identify Failed Samples

We only apply RAG to samples that **failed runtime** in the SFT pre-test. These are the samples the fine-tuned model struggled with and may benefit from retrieval augmentation.

In [ ]:
# Load the smoke report from SFT pre-test
with open(SMOKE_REPORT_PATH, "r", encoding="utf-8") as f:
    smoke_report = json.load(f)

print(f"Loaded smoke report with {len(smoke_report)} entries\n")

# Identify failed sample IDs
# Status can be: "pass", "timeout", "runtime_error", "error", etc.
failed_ids = {
    entry["id"]
    for entry in smoke_report
    if entry.get("status") != "pass"
}

print(f"Failed samples in pre-test: {len(failed_ids)}")

# Get test set IDs
test_ids = {sample["id"] for sample in test_set}

# RAG targets = samples in test set that failed
rag_target_ids = failed_ids & test_ids
print(f"RAG targets (failed ∩ test set): {len(rag_target_ids)}")

# Build list of samples to process
rag_samples = [s for s in test_set if s["id"] in rag_target_ids]
rag_samples.sort(key=lambda x: x["id"])

print(f"\nSamples to process with RAG: {len(rag_samples)}")
for s in rag_samples[:10]:
    print(f"  - {s['id']}")
if len(rag_samples) > 10:
    print(f"  ... and {len(rag_samples) - 10} more")

## 5 — Download & Load Fine-Tuned Model

Download the GGUF model from HuggingFace Hub and load it using `llama-cpp-python` via LangChain's `LlamaCpp` wrapper.

In [ ]:
# Download the GGUF model from HuggingFace Hub if not already present
if not LOCAL_MODEL_PATH.exists():
    print(f"Downloading model from {HF_MODEL_REPO}...")
    downloaded_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=HF_MODEL_FILE,
        local_dir=str(MODEL_DIR),
        local_dir_use_symlinks=False
    )
    print(f"Downloaded to: {downloaded_path}")
else:
    print(f"Model already exists at: {LOCAL_MODEL_PATH}")

# Verify model file size
model_size_gb = LOCAL_MODEL_PATH.stat().st_size / (1024**3)
print(f"Model size: {model_size_gb:.2f} GB")

In [ ]:
# Initialize LlamaCpp model via LangChain
# Using streaming callback for real-time output during inference

callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

llm = LlamaCpp(
    model_path=str(LOCAL_MODEL_PATH),
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    callback_manager=callback_manager,
    verbose=False,  # reduce logging noise
)

print(f"\n✓ LlamaCpp model loaded successfully")
print(f"  Context window: {N_CTX}")
print(f"  GPU layers: {N_GPU_LAYERS}")
print(f"  Temperature: {TEMPERATURE}")

## 6 — Load Vector Store & Reranker

Connect to the pre-built ChromaDB containing library documentation embeddings, and initialize the cross-encoder reranker for result refinement.

In [ ]:
# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"✓ Connected to ChromaDB collection: {COLLECTION_NAME}")
print(f"  Documents in collection: {collection.count()}")

# Initialize embedding model (same as used for indexing)
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5")

# Initialize cross-encoder reranker
reranker = CrossEncoder(RERANKER_MODEL)
print(f"✓ Reranker loaded: {RERANKER_MODEL}")

## 7 — RAG Helper Functions

Define retrieval utilities:
1. **`retrieve_for_library`**: Query ChromaDB for relevant documentation snippets
2. **`rerank_and_filter`**: Apply cross-encoder reranking and score gating
3. **`build_rag_context`**: Orchestrate retrieval across multiple libraries

In [ ]:
def retrieve_for_library(query: str, library: str, n_results: int = N_RETRIEVE) -> list[dict]:
    """
    Retrieve relevant documentation chunks from ChromaDB for a specific library.
    
    Args:
        query: The search query (typically the error message)
        library: Library name to filter by (e.g., "pandas", "sklearn")
        n_results: Number of results to retrieve
    
    Returns:
        List of dicts with 'text', 'metadata', and 'distance'
    """
    # Embed the query
    query_embedding = embedder.encode(query, normalize_embeddings=True).tolist()
    
    # Query ChromaDB with library filter
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where={"library": library},
        include=["documents", "metadatas", "distances"]
    )
    
    # Format results
    docs = []
    for i, doc in enumerate(results["documents"][0]):
        docs.append({
            "text": doc,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })
    
    return docs


def rerank_and_filter(query: str, docs: list[dict], top_k: int = N_RERANK) -> list[dict]:
    """
    Apply cross-encoder reranking and filter by score threshold.
    
    Args:
        query: The search query
        docs: List of document dicts from retrieve_for_library
        top_k: Number of top results to keep after reranking
    
    Returns:
        Filtered and sorted list of document dicts with 'rerank_score'
    """
    if not docs:
        return []
    
    # Prepare pairs for cross-encoder
    pairs = [[query, doc["text"]] for doc in docs]
    
    # Get reranking scores
    scores = reranker.predict(pairs)
    
    # Add scores to docs
    for doc, score in zip(docs, scores):
        doc["rerank_score"] = float(score)
    
    # Sort by score descending and filter
    docs_sorted = sorted(docs, key=lambda x: x["rerank_score"], reverse=True)
    docs_filtered = [d for d in docs_sorted if d["rerank_score"] >= MIN_RERANKER_SCORE]
    
    return docs_filtered[:top_k]


def build_rag_context(error_text: str, libraries: list[str]) -> tuple[str, list[dict]]:
    """
    Build RAG context by retrieving and reranking across multiple libraries.
    
    Args:
        error_text: The error message to use as query
        libraries: List of library names to search
    
    Returns:
        Tuple of (formatted context string, raw retrieved docs)
    """
    # Truncate query if too long
    query = error_text[:MAX_QUERY]
    
    all_docs = []
    
    # Retrieve and rerank for each library
    for lib in libraries:
        docs = retrieve_for_library(query, lib)
        reranked = rerank_and_filter(query, docs)
        all_docs.extend(reranked)
    
    # Sort all docs by rerank score
    all_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
    
    # Build context string with character limit
    context_parts = []
    total_chars = 0
    
    for doc in all_docs:
        chunk = doc["text"]
        if total_chars + len(chunk) > MAX_CTX_CHARS:
            break
        context_parts.append(chunk)
        total_chars += len(chunk)
    
    context_str = "\n\n---\n\n".join(context_parts) if context_parts else ""
    
    return context_str, all_docs


# Test retrieval with a sample query
test_result = retrieve_for_library("SettingWithCopyWarning", "pandas", n_results=2)
print(f"Test retrieval: {len(test_result)} docs retrieved for pandas")
if test_result:
    print(f"  First doc preview: {test_result[0]['text'][:100]}...")

## 8 — Document Summarization

Use the SFT model to summarize retrieved documentation into concise, actionable snippets before adding to the prompt.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARIZATION PROMPT
# ══════════════════════════════════════════════════════════════════════════════

SUMMARIZE_PROMPT = """Summarize the following documentation snippets into 2-3 concise sentences that are most relevant for fixing a Python error.
Focus on: key API usage, common pitfalls, and correct parameter usage.

Documentation:
{docs}

Concise Summary:"""


def summarize_rag_docs(docs_text: str, max_summary_tokens: int = 200) -> str:
    """
    Summarize retrieved documentation using the SFT model.
    
    Args:
        docs_text: Raw documentation text to summarize
        max_summary_tokens: Maximum tokens for the summary
    
    Returns:
        Summarized documentation string
    """
    if not docs_text.strip():
        return ""
    
    # Build the summarization prompt
    prompt = SUMMARIZE_PROMPT.format(docs=docs_text[:2000])  # Limit input
    
    try:
        # Use the LLM directly for summarization
        summary = llm.invoke(prompt, max_tokens=max_summary_tokens)
        # Clean up the output
        summary = summary.strip()
        # Remove any markdown code fences if present
        summary = re.sub(r'^```.*?\n', '', summary)
        summary = re.sub(r'\n```$', '', summary)
        return summary.strip()
    except Exception as e:
        print(f"  [WARN] Summarization failed: {e}")
        # Fallback: return truncated original
        return docs_text[:500] + "..." if len(docs_text) > 500 else docs_text


def build_rag_context_with_summary(
    error_text: str, 
    libraries: list[str],
    summarize: bool = True
) -> tuple[str, str, list[dict]]:
    """
    Build RAG context with optional summarization.
    
    Args:
        error_text: The error message to use as query
        libraries: List of library names to search
        summarize: Whether to summarize the retrieved docs
    
    Returns:
        Tuple of (raw context string, summarized context, raw retrieved docs)
    """
    # Get raw RAG context
    raw_context, all_docs = build_rag_context(error_text, libraries)
    
    if not raw_context:
        return "", "", all_docs
    
    # Summarize if requested
    if summarize and raw_context:
        print("  Summarizing retrieved docs...")
        summarized = summarize_rag_docs(raw_context)
    else:
        summarized = raw_context
    
    return raw_context, summarized, all_docs


print("✓ Summarization functions defined")

## 9 — LangChain Chain Setup

Create a `PromptTemplate` and `LLMChain` for the RAG-augmented code generation task.

**Components:**
- **System prompt**: Defines the model's role as a Python bug-fixer with structured output format
- **User prompt**: Contains buggy code, error message, and summarized documentation context
- **Output format**: Uses `<correct_code>` XML tags for reliable parsing

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SYSTEM PROMPT (matches baseline format for SFT model)
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You fix Python programs.
Return EXACTLY this format and nothing else:
<correct_code>
...full corrected python code...
</correct_code>
<error_type>
...one short line describing the bug type...
</error_type>"""

# ══════════════════════════════════════════════════════════════════════════════
# USER PROMPT TEMPLATE (with RAG context)
# ══════════════════════════════════════════════════════════════════════════════

USER_PROMPT_TEMPLATE = """Fix this Python code based on the runtime error.

Traceback:
{error_message}

Code:
{buggy_code}

Reference docs:
{context}"""

# ══════════════════════════════════════════════════════════════════════════════
# FULL PROMPT TEMPLATE (System + User combined for LlamaCpp)
# ══════════════════════════════════════════════════════════════════════════════
# Note: LlamaCpp doesn't have native chat template support like OpenAI,
# so we combine system + user prompts into a single prompt.

RAG_PROMPT_TEMPLATE = f"""{SYSTEM_PROMPT}


{{user_prompt}}"""print(f"\n✓ Using structured output format with <correct_code> tags")

print(SYSTEM_PROMPT)

print(f"\nSystem Prompt:")

def build_full_prompt(print("✓ LangChain RAG chain created")

    buggy_code: str,

    error_message: str,)

    summarized_context: str    verbose=False

) -> str:    prompt=prompt_template,

    """    llm=llm,

    Build the complete prompt with system instructions and user content.rag_chain = LLMChain(

    # Create the LLMChain

    Args:

        buggy_code: The original buggy code)

        error_message: The error/traceback text    template=RAG_PROMPT_TEMPLATE

        summarized_context: Summarized RAG documentation    input_variables=["user_prompt"],

    prompt_template = PromptTemplate(

    Returns:# Create LangChain components

        Complete prompt string for the model

    """

    # Build user prompt    return full_prompt

    if summarized_context.strip():    full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"

        user_prompt = USER_PROMPT_TEMPLATE.format(    # Combine with system prompt

            buggy_code=buggy_code,    

            error_message=error_message,{buggy_code}"""

            context=summarized_contextCode:

        )

    else:{error_message}

        # No RAG context availableTraceback:

        user_prompt = f"""Fix this Python code based on the runtime error.

## 10 — Output Parsing & Similarity

Helper functions to extract Python code from model output and compute similarity to reference solutions.

3. Raw text fallback

**Parsing priority:**2. Markdown code blocks (```python)
1. `<correct_code>` XML tags (preferred format)

In [ ]:
def extract_python_code(text: str) -> str:
    """
    Extract Python code from model output.
    Priority: <correct_code> tags > markdown blocks > raw text.
    """
    text = (text or "").strip()
    
    # Priority 1: Try <correct_code> XML tags (our preferred format)
    tag_match = re.search(r"<correct_code>\s*(.*?)\s*</correct_code>", text, flags=re.DOTALL | re.IGNORECASE)
    if tag_match:
        code = tag_match.group(1).strip()
        # Strip any markdown fences inside the tag
        code = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", code)
        code = re.sub(r"\s*```$", "", code)
        return code.strip()
    
    # Priority 2: Try markdown code blocks
    code_block_pattern = r"```(?:python)?\s*([\s\S]*?)```"
    matches = re.findall(code_block_pattern, text, re.IGNORECASE)
    if matches:
        # Return the last code block (usually the fixed code)
        return matches[-1].strip()
    
    # Priority 3: Return text as-is (trimmed)
    return text.strip()


def extract_error_type(text: str) -> str:
    """
    Extract error type description from <error_type> tags.
    """
    text = (text or "").strip()
    match = re.search(r"<error_type>\s*(.*?)\s*</error_type>", text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""


def calculate_similarity(code1: str, code2: str) -> float:
    """
    Calculate normalized sequence similarity between two code strings.
    Uses SequenceMatcher for character-level comparison.
    """
    if not code1 or not code2:
        return 0.0
    return difflib.SequenceMatcher(None, code1, code2).ratio()


def get_libraries_from_sample(sample: dict) -> list[str]:
    """
    Extract library names from a dataset sample.
    Looks for 'libraries' field or infers from imports.
    """
    # Try explicit libraries field
    if "libraries" in sample:
        libs = sample["libraries"]
        if isinstance(libs, str):
            return [libs]
        return list(libs)
    
    # Try to infer from buggy code imports
    code = sample.get("buggy_code", "")

    common_libs = ["pandas", "numpy", "sklearn", "matplotlib", "tensorflow", "keras", "torch"]print(extracted[:100] + "..." if len(extracted) > 100 else extracted)

    found = []print("Test extraction:")

    for lib in common_libs:extracted = extract_python_code(test_output)

        if lib in code or f"import {lib}" in code:"""

            found.append(lib)```

    print(df)

    return found if found else ["pandas"]  # default to pandasdf = pd.DataFrame({'a': [1,2,3]})

import pandas as pd

```python

# Test the extractiontest_output = """Here is the fixed code:

## 11 — RAG Inference Loop

Process each failed sample through the RAG pipeline:
1. Extract libraries from sample
2. Build RAG context from error message
3. **Summarize** retrieved docs using the SFT model
4. Run the LangChain chain with system prompt
5. Extract corrected code from `<correct_code>` tags

In [ ]:
results = []
total = len(rag_samples)

print(f"Starting RAG inference on {total} samples\n")
print("=" * 80)

for idx, sample in enumerate(rag_samples):
    sample_id = sample["id"]
    print(f"\n[{idx+1}/{total}] Processing: {sample_id}")
    print("-" * 60)
    
    # Get sample data
    buggy_code = sample.get("buggy_code", "")
    error_message = sample.get("error_message", "")
    correct_code = sample.get("correct_code", "")
    task_description = sample.get("task", "")
    
    # Get libraries for this sample
    libraries = get_libraries_from_sample(sample)
    print(f"Libraries: {libraries}")
    
    # Build RAG context WITH SUMMARIZATION
    raw_context, summarized_context, retrieved_docs = build_rag_context_with_summary(
        error_message, libraries, summarize=True
    )
    print(f"Retrieved {len(retrieved_docs)} docs")
    print(f"  Raw context: {len(raw_context)} chars")
    print(f"  Summarized:  {len(summarized_context)} chars")
    
    # Build the full prompt with system prompt
    user_prompt = USER_PROMPT_TEMPLATE.format(
        buggy_code=buggy_code,
        error_message=error_message,
        context=summarized_context if summarized_context else "No relevant documentation found."
    )
    
    # Run the LangChain chain
    try:
        print("Running inference...")
        chain_output = rag_chain.invoke({"user_prompt": user_prompt})
        raw_output = chain_output.get("text", "")
        
    except Exception as e:
        print(f"ERROR: {e}")
        raw_output = ""
    
    # Extract code from output (handles <correct_code> tags)
    predicted_code = extract_python_code(raw_output)
    error_type = extract_error_type(raw_output)
    
    # Calculate similarity to reference
    sim_to_ref = calculate_similarity(predicted_code, correct_code)
    print(f"Similarity to reference: {sim_to_ref:.3f}")
    if error_type:
        print(f"  Detected error type: {error_type}")
    
    # Build result record
    result = {
        "id": sample_id,
        "task": task_description,
        "libraries": libraries,
        "buggy_code": buggy_code,
        "error_message": error_message,
        "correct_code": correct_code,
        "raw_rag_context": raw_context,
        "summarized_context": summarized_context,
        "raw_output": raw_output,
        "predicted_code": predicted_code,
        "error_type_detected": error_type,
        "sim_to_ref": sim_to_ref,
        "n_docs_retrieved": len(retrieved_docs)
    }
    results.append(result)
    
    # Save individual output file
    output_dir = OUT_DIR / sample_id
    output_dir.mkdir(parents=True, exist_ok=True)
    

    # Save predicted codeprint(f"Completed RAG inference on {len(results)} samples")

    (output_dir / "predicted.py").write_text(predicted_code, encoding="utf-8")print("\n" + "=" * 80)

    

    # Save metadata    print(f"✓ Saved to {output_dir}")

    meta = {k: v for k, v in result.items() if k not in ["predicted_code", "buggy_code", "correct_code"]}    
    (output_dir / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

## 12 — Save Summary Results

Export all results to a JSON file for analysis.

In [ ]:
# Save all results to a summary JSON
summary_path = OUT_DIR / "summary_rag_sft.json"

summary_data = {
    "model": HF_MODEL_REPO,
    "total_samples": len(results),
    "timestamp": datetime.now().isoformat(),
    "config": {
        "n_ctx": N_CTX,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "n_retrieve": N_RETRIEVE,
        "n_rerank": N_RERANK,
        "min_reranker_score": MIN_RERANKER_SCORE,
        "max_ctx_chars": MAX_CTX_CHARS
    },
    "results": results
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=2)

print(f"✓ Summary saved to: {summary_path}")
print(f"  Total results: {len(results)}")

## 13 — Results Summary

Display statistics about the RAG inference results.

In [ ]:
# Calculate statistics
similarities = [r["sim_to_ref"] for r in results]
parseable = [r for r in results if r["predicted_code"].strip()]

print("=" * 60)
print("RAG INFERENCE SUMMARY (SFT Model)")
print("=" * 60)
print(f"\nModel: {HF_MODEL_REPO}")
print(f"Samples processed: {len(results)}")
print(f"Parseable outputs: {len(parseable)} ({100*len(parseable)/len(results):.1f}%)" if results else "")

if similarities:
    print(f"\nSimilarity to Reference:")
    print(f"  Mean:   {sum(similarities)/len(similarities):.3f}")
    print(f"  Min:    {min(similarities):.3f}")
    print(f"  Max:    {max(similarities):.3f}")
    
    # Distribution
    high_sim = sum(1 for s in similarities if s >= 0.8)
    med_sim = sum(1 for s in similarities if 0.5 <= s < 0.8)
    low_sim = sum(1 for s in similarities if s < 0.5)
    print(f"\nSimilarity Distribution:")
    print(f"  High (≥0.8): {high_sim} ({100*high_sim/len(similarities):.1f}%)")
    print(f"  Med (0.5-0.8): {med_sim} ({100*med_sim/len(similarities):.1f}%)")
    print(f"  Low (<0.5): {low_sim} ({100*low_sim/len(similarities):.1f}%)")

print("\n" + "=" * 60)

---

## 14 — Evaluation: Syntax Check

Verify that generated code compiles without syntax errors using `py_compile`.

In [ ]:
def check_syntax(code: str) -> tuple[bool, str]:
    """
    Check if Python code has valid syntax.
    Returns (is_valid, error_message).
    """
    if not code.strip():
        return False, "Empty code"
    
    try:
        compile(code, "<string>", "exec")
        return True, ""
    except SyntaxError as e:
        return False, f"Line {e.lineno}: {e.msg}"


# Check syntax for all results
syntax_results = []

for r in results:
    sample_id = r["id"]
    code = r["predicted_code"]
    is_valid, error = check_syntax(code)
    
    syntax_results.append({
        "id": sample_id,
        "valid": is_valid,
        "error": error
    })

# Summary
valid_count = sum(1 for s in syntax_results if s["valid"])
print(f"Syntax Check Results:")
print(f"  Valid:   {valid_count}/{len(syntax_results)} ({100*valid_count/len(syntax_results):.1f}%)")
print(f"  Invalid: {len(syntax_results) - valid_count}")

# Show invalid samples
invalid = [s for s in syntax_results if not s["valid"]]
if invalid:
    print("\nInvalid samples:")
    for s in invalid[:10]:
        print(f"  - {s['id']}: {s['error']}")
    if len(invalid) > 10:
        print(f"  ... and {len(invalid) - 10} more")

## 15 — Fast Eval Patching

Reduce epochs and iterations for faster runtime testing.

In [ ]:
# Samples that should NOT have epochs patched (use original values)
NO_EPOCH_PATCH = {"081"}

def patch_fast_eval(code: str, skip_epoch_patch: bool = False) -> str:
    """
    Patch code for faster evaluation by reducing epochs and iterations.
    
    Args:
        code: The Python code to patch
        skip_epoch_patch: If True, skip epoch-related patches (for samples that need original values)
    
    Returns:
        Patched code string
    """
    # Reduce epochs (unless skipped)
    if not skip_epoch_patch:
        code = re.sub(r'epochs\s*=\s*\d+', 'epochs=5', code)
        code = re.sub(r'n_iter\s*=\s*\d+', 'n_iter=5', code)
        code = re.sub(r'max_iter\s*=\s*\d+', 'max_iter=100', code)
    
    # Reduce verbosity (always apply)
    code = re.sub(r'verbose\s*=\s*[12]', 'verbose=0', code)
    
    # Force CPU if configured
    if FORCE_CPU:
        # Add CPU-only environment variables at the top
        cpu_header = '''import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
'''
        if "CUDA_VISIBLE_DEVICES" not in code:
            code = cpu_header + code
    
    return code


# Test the patching
test_code = "model.fit(X, y, epochs=100, verbose=1)"
patched = patch_fast_eval(test_code)
print(f"Original: {test_code}")
print(f"Patched:  {patched}")

## 16 — Runtime Smoke Test

Run each generated script in an isolated subprocess to check for runtime errors.

In [ ]:
def run_script(code: str, timeout: int = TIMEOUT) -> dict:
    """
    Run Python code in a subprocess with timeout.
    
    Returns:
        Dict with 'status', 'stdout', 'stderr', 'runtime'
    """
    # Create a temporary file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as f:
        f.write(code)
        temp_path = f.name
    
    start = time.time()
    try:
        result = subprocess.run(
            [sys.executable, temp_path],
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=str(OUT_DIR)  # Run from output directory
        )
        runtime = time.time() - start
        
        if result.returncode == 0:
            return {
                "status": "pass",
                "stdout": result.stdout[:2000],
                "stderr": result.stderr[:2000],
                "runtime": runtime
            }
        else:
            return {
                "status": "runtime_error",
                "stdout": result.stdout[:2000],
                "stderr": result.stderr[:2000],
                "runtime": runtime
            }
    
    except subprocess.TimeoutExpired:
        return {
            "status": "timeout",
            "stdout": "",
            "stderr": f"Timeout after {timeout}s",
            "runtime": timeout
        }
    
    except Exception as e:
        return {
            "status": "error",
            "stdout": "",
            "stderr": str(e),
            "runtime": time.time() - start
        }
    
    finally:
        # Clean up temp file
        try:
            os.unlink(temp_path)
        except:
            pass


# Run smoke tests on all valid syntax samples
smoke_results = []
valid_samples = [r for r, s in zip(results, syntax_results) if s["valid"]]

print(f"Running smoke test on {len(valid_samples)} samples with valid syntax")
print("=" * 60)

for i, r in enumerate(valid_samples):
    sample_id = r["id"]
    
    # Extract sample index for NO_EPOCH_PATCH check
    idx_str = sample_id.split("_")[0] if "_" in sample_id else sample_id[:3]
    skip_epoch = idx_str in NO_EPOCH_PATCH
    
    # Patch for fast eval
    patched_code = patch_fast_eval(r["predicted_code"], skip_epoch_patch=skip_epoch)
    
    print(f"\n[{i+1}/{len(valid_samples)}] {sample_id}", end="")
    if skip_epoch:
        print(" (original epochs)", end="")
    print("...")
    
    result = run_script(patched_code)
    smoke_results.append({
        "id": sample_id,
        **result
    })
    
    status_icon = "✓" if result["status"] == "pass" else "✗"
    print(f"  {status_icon} {result['status']} ({result['runtime']:.1f}s)")
    
    if result["status"] != "pass" and result["stderr"]:
        # Show first line of error
        error_line = result["stderr"].split("\n")[-2] if "\n" in result["stderr"] else result["stderr"]
        print(f"    Error: {error_line[:100]}")

print("\n" + "=" * 60)

## 17 — Evaluation Summary

Aggregate evaluation results and save smoke test report.

In [ ]:
# Count results by status
status_counts = {}
for sr in smoke_results:
    status = sr["status"]
    status_counts[status] = status_counts.get(status, 0) + 1

passed = status_counts.get("pass", 0)
total_tested = len(smoke_results)

print("=" * 60)
print("EVALUATION SUMMARY (SFT + RAG)")
print("=" * 60)
print(f"\nTotal RAG samples: {len(results)}")
print(f"Valid syntax: {len(valid_samples)}")
print(f"Tested: {total_tested}")
print(f"\nRuntime Results:")
for status, count in sorted(status_counts.items()):
    pct = 100 * count / total_tested if total_tested > 0 else 0
    print(f"  {status}: {count} ({pct:.1f}%)")

if total_tested > 0:
    print(f"\nPass Rate: {passed}/{total_tested} ({100*passed/total_tested:.1f}%)")

# Save smoke report
smoke_report_out = OUT_DIR / "smoke_report_rag_sft.json"
with open(smoke_report_out, "w", encoding="utf-8") as f:
    json.dump(smoke_results, f, indent=2)
print(f"\n✓ Smoke report saved to: {smoke_report_out}")

print("=" * 60)

## 18 — Comparison: SFT vs SFT+RAG

Compare the RAG-enhanced results with the original SFT model pre-test results.

In [ ]:
# Build lookup of RAG smoke results
rag_smoke_lookup = {sr["id"]: sr["status"] for sr in smoke_results}

# Build lookup of original smoke results
original_lookup = {entry["id"]: entry.get("status", "unknown") for entry in smoke_report}

# Calculate overall statistics
total_samples = len(dataset_full)
original_passed = sum(1 for entry in smoke_report if entry.get("status") == "pass")
original_failed = total_samples - original_passed

# Count how many RAG samples now pass
rag_fixed = 0
still_failed = 0
newly_broken = 0  # Should not happen, but track it

for sr in smoke_results:
    sample_id = sr["id"]
    rag_status = sr["status"]
    orig_status = original_lookup.get(sample_id, "unknown")
    
    if orig_status != "pass" and rag_status == "pass":
        rag_fixed += 1
    elif orig_status != "pass" and rag_status != "pass":
        still_failed += 1

# Final counts
final_passed = original_passed + rag_fixed

print("=" * 70)
print("OVERALL COMPARISON: SFT vs SFT+RAG")
print("=" * 70)

print(f"\n{'Metric':<35} {'Count':>10} {'Percentage':>15}")
print("-" * 70)
print(f"{'Total samples in dataset':<35} {total_samples:>10}")
print(f"{'Original SFT passed':<35} {original_passed:>10} {100*original_passed/total_samples:>14.1f}%")
print(f"{'Original SFT failed':<35} {original_failed:>10} {100*original_failed/total_samples:>14.1f}%")
print("-" * 70)
print(f"{'RAG targets (failed samples)':<35} {len(results):>10}")
print(f"{'RAG attempts with valid syntax':<35} {len(valid_samples):>10}")
print(f"{'RAG fixed (now pass)':<35} {rag_fixed:>10}")
print(f"{'Still failed after RAG':<35} {still_failed:>10}")
print("-" * 70)
print(f"{'FINAL: Passed after SFT+RAG':<35} {final_passed:>10} {100*final_passed/total_samples:>14.1f}%")
print("=" * 70)

# Improvement summary
improvement = final_passed - original_passed
improvement_pct = 100 * improvement / original_failed if original_failed > 0 else 0
overall_improvement = 100 * (final_passed / total_samples) - 100 * (original_passed / total_samples)

print(f"\n📊 IMPROVEMENT SUMMARY:")
print(f"   RAG fixed {rag_fixed} samples that SFT alone couldn't handle")
print(f"   {improvement_pct:.1f}% of failed samples recovered by RAG")
print(f"   Overall pass rate: {100*original_passed/total_samples:.1f}% → {100*final_passed/total_samples:.1f}% (+{overall_improvement:.1f}%)")
print("=" * 70)